In [2]:
import ast
import pandas as pd

df = pd.read_csv("../data/billboard_top_100_1946_2022_lyrics-edited.csv")

# Parse the stringified lists back into normal text
df["artist"] = df["Artist Names"].apply(lambda s: ", ".join(ast.literal_eval(s)))
df["lyrics"] = df["Lyrics"].apply(lambda s: " ".join(ast.literal_eval(s)))

# Clean titles like "Prisoner of Love - Remastered - 1992"
df["title"] = df["Song"].str.replace(r"\s+-\s+remaster(ed)?.*$", "", regex=True, case=False)

# Rename the metadata columns and drop the originals
df = df.rename(columns={"Hot100 Ranking Year": "year", "Hot100 Rank": "rank"})
df = df[["title", "artist", "year", "rank", "lyrics"]]

# Drop songs with no lyrics and exact duplicates
df = df[df["lyrics"].str.strip() != ""]
df = df.drop_duplicates(subset=["title", "artist", "year"]).reset_index(drop=True)

print(df.shape)
print(df.head(3))
print(df["lyrics"].str.split().str.len().describe())

(6027, 5)
              title         artist  year  rank  \
0  Prisoner of Love     perry como  1946     1   
1   To Each His Own    eddy howard  1946     2   
2         The Gypsy  the ink spots  1946     3   

                                              lyrics  
0  someone that i belong to doesnt belong to me s...  
1  a rose must remain with the sun and the rain o...  
2  in a quaint caravan theres a lady they call th...  
count    6027.000000
mean      337.790443
std       179.113234
min         3.000000
25%       209.000000
50%       297.000000
75%       419.000000
max      1282.000000
Name: lyrics, dtype: float64


In [3]:
short = df[df["lyrics"].str.split().str.len() < 30]
print(len(short))
df = df[df["lyrics"].str.split().str.len() >= 30].reset_index(drop=True)

5


In [4]:
df.to_csv("../data/songs_clean.csv", index=False)